---
title: Migration analysis
date: now
author: Jan Cap
---

In [ ]:
from geoscore_de.data_flow.features.municipality import MunicipalityFeature

municipalities = MunicipalityFeature("../../data/raw/municipalities_2022.csv")
municipalities = municipalities.load()
municipalities

## Feature alignment

Align migration features with the municipality dataset and explore missing rows.

In [ ]:
from geoscore_de.data_flow.features.migration import MigrationFeature

migration = MigrationFeature(
    "../../data/raw/features/12711-91-01-5-migration.csv",
    "../../data/tform/features/migration.csv",
    "../../data/raw/municipalities_2022.csv",
).load()
migration

Data contain data for multiple years. From 2020 to 2024. Data were pivoted to have one col per year and statistic (in, out migration). 

In [ ]:
from geoscore_de.data_flow.features.migration import MigrationFeature

migration = MigrationFeature(
    "../../data/raw/features/12711-91-01-5-migration.csv",
    "../../data/tform/features/migration.csv",
    "../../data/raw/municipalities_2022.csv",
).load_transform()

In [ ]:
migration = migration.merge(municipalities, on="AGS", how="outer", indicator=True)
migration

In [ ]:
migration["_merge"].value_counts()

All municipalities are present in the migration feature, but there are 366 rows in the migration feature that do not have a match in the municipality dataset. This is due to the fact that the migration feature contains also data for bigger administrative units (e.g. districts) than municipalities. These rows will be dropped in the next steps, as they are not relevant for the analysis.



In transformation the net migration was added as a new feature, which is the difference between in and out migration. Also all migration features were normalized by the population of the municipality to have comparable values across municipalities of different sizes.

In [ ]:
import plotnine as gg

# show rate of in/out migration as histogram
(
    gg.ggplot(migration)
    + gg.aes(x="in_migration_2024")
    + gg.geom_histogram(bins=30)
    + gg.labs(title="In Ratio Distribution (in/Persons)")
)

In [ ]:
import plotnine as gg

# show rate of in/out migration as histogram
(
    gg.ggplot(migration)
    + gg.aes(x="out_migration_2024")
    + gg.geom_histogram(bins=30)
    + gg.labs(title="Out Ratio Distribution (out/Persons)")
)

In [ ]:
import plotnine as gg

# show rate of in/out migration as histogram
(
    gg.ggplot(migration)
    + gg.aes(x="net_migration_2024")
    + gg.geom_histogram(bins=30)
    + gg.labs(title="Net Migration Ratio Distribution ((in-out)/Persons)")
)

In [ ]:
# show top 10 municipalities with highest in-migration
migration[
    ["in_migration_2024", "out_migration_2024", "net_migration_2024", "AGS", "Municipality", "Persons"]
].sort_values("in_migration_2024", ascending=False).head(10)

There is definitely some bug in data. Municipality with population of 600 people cannot have 7 times more in and out migration than its population.
We should use only net migration feature, which is the difference between in and out migration.

In [ ]:
# show top 10 municipalities with highest net migration
display(
    migration[["in_migration_2024", "out_migration_2024", "net_migration_2024", "AGS", "Municipality", "Persons"]]
    .sort_values("net_migration_2024", ascending=False)
    .head(10)
)
# show top 10 municipalities with lowest net migration
display(
    migration[["in_migration_2024", "out_migration_2024", "net_migration_2024", "AGS", "Municipality", "Persons"]]
    .sort_values("net_migration_2024", ascending=True)
    .head(10)
)

Net migration looks much cleaner. Minicipality with highest net migration has population of 400 and net migration of 60%, which is possible.
Municipality with lowest net migration has population of 75 and net migration of -40%, which is also possible.

## Visualizations

In [ ]:
# now show it on map of germany
from geoscore_de.data_flow.geo import load_geo_data

gdf = load_geo_data("../../data/georef-germany-gemeinde.csv")

In [ ]:
migration = migration.drop(columns=["_merge"])
gdf_merged = gdf.merge(migration, on="AGS", how="left", indicator=True)

In [ ]:
# print count of missing values
gdf_merged["_merge"].value_counts()

In [ ]:
import matplotlib.pyplot as plt

gdf_merged.plot(
    column="net_migration_2024",
    legend=True,
    cmap="coolwarm",
    figsize=(10, 10),
    vmin=-0.05,
    vmax=0.05,
    missing_kwds={"color": "gray"},
)
plt.title("Proportional net migration in German Municipalities 2024")
plt.show()

According to the data there are more in-migration than out-migration in 2024. The most visible differences are in small municipalities, which is expected. Espetially visible is it in Rhineland-Palatinate, which is separated to smaller municipalities than in other states. In some municipalities there is more than 10% of population that migrated in or out, which is a huge number.

Lets compare migration update from 2020 with the one from 2024.

In [ ]:
gdf_merged["update_20_24"] = gdf_merged["net_migration_2024"] - gdf_merged["net_migration_2020"]
gdf_merged["update_20_24"]

In [ ]:
import matplotlib.pyplot as plt

# use colormap that center is 1 (in_out_ratio = 1 means equal in and out migration)
gdf_merged.plot(
    column="update_20_24",
    legend=True,
    cmap="coolwarm",
    figsize=(10, 10),
    vmin=-0.05,
    vmax=0.05,
    missing_kwds={"color": "gray"},
)
plt.title("Difference in proportional net migration between 2020 and 2024")
plt.show()

We dont see any trend or pattern in the migration difference between 2020 and 2024. There are some municipalities with higher net migration in 2024 than in 2020, but there are also some with lower net migration. Overall it seems that there is no significant change in migration patterns between these two years.